<a href="https://colab.research.google.com/github/KruthikaVirupakshappa/DATA266-Lab1-Pair37/blob/jiayi-work/task1_llm/jiayi/src/Task1_GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
SID4         =  4253 # last four digits of your SJSU ID
SEED         = SID4        # torch.manual_seed, np.random.seed, random_state
SLICE        = SID4 % 1000   # start index of your dataset subset
HP_ID        = SID4 % 6      # your assigned hyperparameter arm, Table 0.1
CLS_A, CLS_B = (SID4 % 10), ((SID4 // 10) % 10)   # your two focus classes

In [2]:
print(f"SID4: {SID4}, SEED: {SEED}, SLICE: {SLICE}, HP_ID: {HP_ID}, CLS_A, CLS_B: {CLS_A}, {CLS_B}")

SID4: 4253, SEED: 4253, SLICE: 253, HP_ID: 5, CLS_A, CLS_B: 3, 5


## Task 1
### data preparation

In [35]:
import random
import numpy as np
import torch
from pathlib import Path
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
import math
import torch.nn as nn
import torch.nn.functional as F

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [49]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [21]:
# load shared data from download_tinystories.py
from datasets import load_from_disk
from pathlib import Path
from datasets import load_dataset

dataset_folder = Path("task1_llm/data/tinystories")
data = load_from_disk(str(dataset_folder))

# inspect the original dataset
print(data)
print(data.keys())

# only use the original training set for training and validation
original_train = data["train"]

# use SEED to randomize original_train
shuffled_train = original_train.shuffle(seed=SEED)

# split the shuffled train
my_train = shuffled_train.select(range(100_000))
my_val = shuffled_train.select(range(100_000, 110000))

# inspect the splitted dataset by checking shape and a few data
print(len(my_train))
print(len(my_val))
print(my_train.column_names)
print(my_train[0])
print(my_val[0])

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})
dict_keys(['train', 'validation'])
100000
10000
['text']
{'text': 'Once upon a time, there was a little bird named Blue. Blue was very fast and loved to fly high in the sky. One day, Blue met a little girl named Lily. \n\n"Hello, Blue! You can fly very fast!" said Lily. \n\n"Yes, I can fly very fast!" replied Blue, happily. \n\nBut then, a big storm came and Blue got lost. Blue was scared and didn\'t know what to do. \n\n"Can you help me, Lily?" asked Blue. \n\n"Yes, I can help you!" said Lily. "Let\'s look for your history book. Maybe it will help us find your way home." \n\nSo, Lily and Blue searched for Blue\'s history book. They found it and read about all the places Blue had flown before. They followed the book and finally, Blue found his way home. \n\n"Thank you, Lily! You\'re my hero!" said Blue. \

In [24]:
train_text = "\n".join(my_train["text"])
val_text = "\n".join(my_val["text"])

chars = sorted(list(set(train_text)))
vocab_size = len(chars)

char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}

def encode(s):
    return [char_to_idx[c] for c in s]

def decode(ids):
    return "".join(idx_to_char[i] for i in ids)

unseen_chars = set(val_text) - set(chars)
print("Unseen validation chars:", unseen_chars)

Unseen validation chars: set()


In [25]:
train_data = torch.tensor(encode(train_text), dtype=torch.long)
val_data = torch.tensor(encode(val_text), dtype=torch.long)

print("Vocab size:", vocab_size)
print("Train characters:", len(train_data))
print("Val characters:", len(val_data))

Vocab size: 152
Train characters: 89734384
Val characters: 8964594


In [36]:
class CharDataset(Dataset):
    def __init__(self, data, seq_len):
        self.data = data
        self.seq_len = seq_len

    def __len__(self):
        return (len(self.data) - 1) // self.seq_len

    def __getitem__(self, idx):
        start = idx * self.seq_len

        x = self.data[start : start + self.seq_len]
        y = self.data[start + 1 : start + self.seq_len + 1]

        return x, y

In [37]:
seq_len = 128
train_dataset = CharDataset(train_data, seq_len)
val_dataset = CharDataset(val_data, seq_len)

print(len(train_dataset))
print(len(val_dataset))

701049
70035


In [38]:
x, y = train_dataset[0]

print(x.shape)
print(y.shape)
print("Input:\n" + decode(x.tolist()))
print("Output:\n" + decode(y.tolist()))

torch.Size([128])
torch.Size([128])
Input:
Once upon a time, there was a little bird named Blue. Blue was very fast and loved to fly high in the sky. One day, Blue met a l
Output:
nce upon a time, there was a little bird named Blue. Blue was very fast and loved to fly high in the sky. One day, Blue met a li


In [40]:
generator = torch.Generator()
generator.manual_seed(SEED)
train_loader = DataLoader(train_dataset, batch_size=32,shuffle=True, generator=generator)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

### GPT model

In [41]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "Embedding dimension must be 0 modulo number of heads."
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, C = x.shape

        QKV = self.qkv(x) # [B, T, 3*d_model]

        QKV = QKV.reshape(B, T, self.num_heads, 3*self.head_dim)
        QKV = QKV.permute(0, 2, 1, 3) # [Batch, Head, SeqLen, Dims]
        Q,K,V = QKV.chunk(3, dim=-1) # each [B, H, T, D]

        # Attention scores between every pair of token
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.head_dim)

        # causal mask
        # prevent token from attending future tokens
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        scores = scores.masked_fill(mask, float('-inf'))

        attn = F.softmax(scores, dim=-1)
        out = attn @ V
        # merge all attention heads back to [B, T, C]
        out = out.permute(0,2,1,3) # [Batch, SeqLen, Head, Dims]
        out = out.reshape(B,T,C)
        out = self.out_proj(out)


        return out


In [44]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, num_heads)
        self.ln2 = nn.LayerNorm(d_model)

        self.ff = nn.Sequential(
            nn.Linear(d_model, 4*d_model),
            nn.GELU(),
            nn.Linear(4*d_model, d_model)
        )

    def forward(self, x):
        # self-attention with residual connection
        x = x + self.attn(self.ln1(x))
        # feedforward network with residual connection
        x = x + self.ff(self.ln2(x))
        return x

In [50]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, d_model=128, seq_len=128, num_heads=4):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.num_layers = 2
        self.seq_len = seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(seq_len, d_model)

        self.block1 = DecoderBlock(d_model, num_heads)
        self.block2 = DecoderBlock(d_model, num_heads)
        self.ln = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, idx):
        B, T = idx.shape
        # Convert token IDs and positions into trainable embeddings
        tok = self.token_emb(idx)
        pos_ids = torch.arange(T, device=idx.device)
        pos = self.pos_emb(pos_ids)

        x = tok + pos
        # two stack decoder blocks
        x = self.block1(x)
        x = self.block2(x)
        x = self.ln(x)
        # project each token representation to vocabulary logits
        logits = self.head(x)

        return logits

model = TinyGPT(vocab_size, seq_len=seq_len).to(device)
print("Model parameters:", sum(p.numel() for p in model.parameters()))
print("Hidden dimension:", model.d_model)
print("Number of heads:", model.num_heads)
print("Number of layers:", model.num_layers)

Model parameters: 452248
Hidden dimension: 128
Number of heads: 4
Number of layers: 2
